In [6]:
import h5py
import awkward as ak
import numpy as np
import os

In [7]:
h5_path = "hgcal_electron_data_0001.h5"
output_dir = "parquet_chunks"

In [ ]:
#Divinding the data into chunks
with h5py.File(h5_path, 'r') as f:

    nhits = np.array(f['nhits'], dtype=np.int32)
    target_all = np.array(f['target'], dtype=np.float32)
    
    #idxs to identify the start and end of each event in the hits arrays
    offsets = np.insert(np.cumsum(nhits), 0, 0)

n_events = len(nhits)
chunk_size = 100000

print(f"Working on: {chunk_size} events...")

for i in range(0, n_events, chunk_size):
    start_ev = i
    end_ev = min(i + chunk_size, n_events)
    
    start_hit = offsets[start_ev]
    end_hit = offsets[end_ev]
    
    print(f"--- Working on events {start_ev} to {end_ev} (Hits: {end_hit - start_hit}) ---")
    
    #Float32 to reduce the file size
    with h5py.File(h5_path, 'r') as f:
        e_chunk = ak.unflatten(f['rechit_energy'][start_hit:end_hit].astype(np.float32), nhits[start_ev:end_ev])
        x_chunk = ak.unflatten(f['rechit_x'][start_hit:end_hit].astype(np.float32), nhits[start_ev:end_ev])
        y_chunk = ak.unflatten(f['rechit_y'][start_hit:end_hit].astype(np.float32), nhits[start_ev:end_ev])
        z_chunk = ak.unflatten(f['rechit_z'][start_hit:end_hit].astype(np.float32), nhits[start_ev:end_ev])
        t_chunk = target_all[start_ev:end_ev]


        chunk_array = ak.zip({
            "energy": e_chunk,
            "x": x_chunk,
            "y": y_chunk,
            "z": z_chunk,
            "target": t_chunk
        })
        
        chunk_name = f"{output_dir}/chunk_{start_ev}.parquet"
        ak.to_parquet(chunk_array, chunk_name)
        
        print(f"Saved: {chunk_name}")

print("\nAll chunks saved.")

Inizio elaborazione a blocchi di 100000 eventi...
--- Elaborazione eventi 0 a 100000 (Hits: 79964243) ---
Salvato: parquet_chunks/chunk_0.parquet
--- Elaborazione eventi 100000 a 200000 (Hits: 79932783) ---
Salvato: parquet_chunks/chunk_100000.parquet
--- Elaborazione eventi 200000 a 300000 (Hits: 79937291) ---
Salvato: parquet_chunks/chunk_200000.parquet
--- Elaborazione eventi 300000 a 400000 (Hits: 79854009) ---
Salvato: parquet_chunks/chunk_300000.parquet
--- Elaborazione eventi 400000 a 500000 (Hits: 80200079) ---
Salvato: parquet_chunks/chunk_400000.parquet
--- Elaborazione eventi 500000 a 600000 (Hits: 79884843) ---
Salvato: parquet_chunks/chunk_500000.parquet
--- Elaborazione eventi 600000 a 648277 (Hits: 38502656) ---
Salvato: parquet_chunks/chunk_600000.parquet

Fatto! Tutti i pezzi sono stati salvati nella cartella 'parquet_chunks'.
